# R1 — Star Schema (claims.analytics)

One governed star schema every downstream tool reads from, built from the trusted layers of Papers 1-2:

* **Sources**: `claims.gold.claims_enriched` (claims), `claims.silver.policy_history` (SCD2 policy versions — the Paper 1 point-in-time rule), `claims.raw.members`, `claims.raw.hospitals`, `claims.raw.plans`.
* **Plan at admission**: `fact_claims.plan_sk` resolves to the policy version in force ON the admission date (`__START_AT <= admission <= __END_AT`), never the current plan.
* **Keys**: surrogate keys are whole numbers, unique, never empty. PK/FK constraints are informational (Unity Catalog does not enforce them), so the final cell re-checks every join resolves by hand.
* **Clustering**: `fact_claims` is clustered by `admission_date_key` so date queries stay fast as it grows.


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS claims.analytics;

-- One row per calendar day from 2025-07-01 to 2026-06-30
CREATE OR REPLACE TABLE claims.analytics.dim_date AS
SELECT
  CAST(date_format(d.full_date, 'yyyyMMdd') AS INT) AS date_key,
  d.full_date,
  YEAR(d.full_date)                              AS year,
  QUARTER(d.full_date)                           AS quarter,
  MONTH(d.full_date)                             AS month,
  CAST(date_trunc('MONTH', d.full_date) AS DATE) AS month_start,
  CAST(date_trunc('WEEK', d.full_date) AS DATE)  AS week_start  -- Monday-starting week
FROM (
  SELECT explode(sequence(DATE'2025-07-01', DATE'2026-06-30', INTERVAL 1 DAY)) AS full_date
) d;

In [0]:
%sql
-- dim_member: no PII columns (name / dob / national_id / phone / email excluded)
CREATE OR REPLACE TABLE claims.analytics.dim_member AS
SELECT
  CAST(ROW_NUMBER() OVER (ORDER BY m.member_id) AS BIGINT) AS member_sk,
  m.member_id, m.gender, m.city, m.state
FROM claims.raw.members m;

-- dim_hospital
CREATE OR REPLACE TABLE claims.analytics.dim_hospital AS
SELECT
  CAST(ROW_NUMBER() OVER (ORDER BY h.hospital_id) AS BIGINT) AS hospital_sk,
  h.hospital_id, h.name AS hospital_name, h.city, h.city_tier, h.state
FROM claims.raw.hospitals h;

In [0]:
%sql
-- One row per policy version (SCD2), with plan attributes at that version
CREATE OR REPLACE TABLE claims.analytics.dim_plan AS
SELECT
  CAST(ROW_NUMBER() OVER (ORDER BY ph.policy_id, ph.__START_AT) AS BIGINT) AS plan_sk,
  ph.member_id, ph.policy_id, ph.plan_code, p.plan_family,
  CAST(p.sum_insured AS INT)           AS sum_insured,
  CAST(p.room_rent_cap_per_day AS INT) AS room_rent_cap_per_day,
  CAST(p.annual_premium AS INT)        AS annual_premium,
  ph.__START_AT AS valid_from,
  ph.__END_AT   AS valid_to,          -- NULL = still open
  ph.__END_AT IS NULL AS is_current
FROM claims.silver.policy_history ph
JOIN claims.raw.plans p ON ph.plan_code = p.plan_code;

In [0]:
%sql
-- One row per claim; plan_sk is the version in force at ADMISSION (never the current plan)
CREATE OR REPLACE TABLE claims.analytics.fact_claims
CLUSTER BY (admission_date_key) AS
SELECT
  e.claim_id,
  dm.member_sk,
  dh.hospital_sk,
  dp.plan_sk,
  CAST(date_format(to_date(e.admission_date), 'yyyyMMdd') AS INT) AS admission_date_key,
  CAST(date_format(to_date(e.submitted_ts),  'yyyyMMdd') AS INT) AS submitted_date_key,
  CAST(date_format(to_date(e.settled_ts),    'yyyyMMdd') AS INT) AS settled_date_key,  -- NULL until settled
  e.claim_type, e.status, e.rejection_code,
  e.is_network_at_admission, e.room_rent_over_cap,
  CAST(ROUND(e.billed_amount,   2) AS DECIMAL(14,2)) AS billed_amount,
  CAST(ROUND(e.approved_amount, 2) AS DECIMAL(14,2)) AS approved_amount,
  e.length_of_stay, e.turnaround_days
FROM claims.gold.claims_enriched e
JOIN claims.silver.policy_history ph
  ON  e.policy_id = ph.policy_id
  AND to_date(e.admission_date) >= ph.__START_AT
  AND (ph.__END_AT IS NULL OR to_date(e.admission_date) <= ph.__END_AT)
JOIN claims.analytics.dim_member   dm ON e.member_id   = dm.member_id
JOIN claims.analytics.dim_hospital dh ON e.hospital_id = dh.hospital_id
JOIN claims.analytics.dim_plan     dp ON dp.policy_id  = ph.policy_id AND dp.valid_from = ph.__START_AT;

In [0]:
%sql
-- Informational constraints (UC does not enforce them; the verify cell re-checks by hand)
ALTER TABLE claims.analytics.dim_date     ALTER COLUMN date_key      SET NOT NULL;
ALTER TABLE claims.analytics.dim_member   ALTER COLUMN member_sk     SET NOT NULL;
ALTER TABLE claims.analytics.dim_hospital ALTER COLUMN hospital_sk   SET NOT NULL;
ALTER TABLE claims.analytics.dim_plan     ALTER COLUMN plan_sk       SET NOT NULL;
ALTER TABLE claims.analytics.fact_claims  ALTER COLUMN claim_id      SET NOT NULL;

ALTER TABLE claims.analytics.fact_claims DROP CONSTRAINT IF EXISTS fact_claims_member_fk;
ALTER TABLE claims.analytics.fact_claims DROP CONSTRAINT IF EXISTS fact_claims_hospital_fk;
ALTER TABLE claims.analytics.fact_claims DROP CONSTRAINT IF EXISTS fact_claims_plan_fk;
ALTER TABLE claims.analytics.fact_claims DROP CONSTRAINT IF EXISTS fact_claims_admission_date_fk;
ALTER TABLE claims.analytics.fact_claims DROP CONSTRAINT IF EXISTS fact_claims_submitted_date_fk;
ALTER TABLE claims.analytics.fact_claims DROP CONSTRAINT IF EXISTS fact_claims_settled_date_fk;

ALTER TABLE claims.analytics.dim_date     DROP CONSTRAINT IF EXISTS dim_date_pk;
ALTER TABLE claims.analytics.dim_member   DROP CONSTRAINT IF EXISTS dim_member_pk;
ALTER TABLE claims.analytics.dim_hospital DROP CONSTRAINT IF EXISTS dim_hospital_pk;
ALTER TABLE claims.analytics.dim_plan     DROP CONSTRAINT IF EXISTS dim_plan_pk;
ALTER TABLE claims.analytics.fact_claims  DROP CONSTRAINT IF EXISTS fact_claims_pk;

ALTER TABLE claims.analytics.dim_date     ADD CONSTRAINT dim_date_pk     PRIMARY KEY (date_key);
ALTER TABLE claims.analytics.dim_member   ADD CONSTRAINT dim_member_pk   PRIMARY KEY (member_sk);
ALTER TABLE claims.analytics.dim_hospital ADD CONSTRAINT dim_hospital_pk PRIMARY KEY (hospital_sk);
ALTER TABLE claims.analytics.dim_plan     ADD CONSTRAINT dim_plan_pk     PRIMARY KEY (plan_sk);
ALTER TABLE claims.analytics.fact_claims  ADD CONSTRAINT fact_claims_pk  PRIMARY KEY (claim_id);

ALTER TABLE claims.analytics.fact_claims ADD CONSTRAINT fact_claims_member_fk         FOREIGN KEY (member_sk)         REFERENCES claims.analytics.dim_member   (member_sk);
ALTER TABLE claims.analytics.fact_claims ADD CONSTRAINT fact_claims_hospital_fk       FOREIGN KEY (hospital_sk)       REFERENCES claims.analytics.dim_hospital (hospital_sk);
ALTER TABLE claims.analytics.fact_claims ADD CONSTRAINT fact_claims_plan_fk           FOREIGN KEY (plan_sk)           REFERENCES claims.analytics.dim_plan     (plan_sk);
ALTER TABLE claims.analytics.fact_claims ADD CONSTRAINT fact_claims_admission_date_fk FOREIGN KEY (admission_date_key) REFERENCES claims.analytics.dim_date    (date_key);
ALTER TABLE claims.analytics.fact_claims ADD CONSTRAINT fact_claims_submitted_date_fk FOREIGN KEY (submitted_date_key) REFERENCES claims.analytics.dim_date    (date_key);
ALTER TABLE claims.analytics.fact_claims ADD CONSTRAINT fact_claims_settled_date_fk   FOREIGN KEY (settled_date_key)   REFERENCES claims.analytics.dim_date    (date_key);

In [0]:
%sql
-- Every surrogate key: whole number, unique, never empty
SELECT 'dim_date' AS tbl, count(*) AS rows_n, count(DISTINCT date_key) AS distinct_keys, sum(CASE WHEN date_key IS NULL THEN 1 ELSE 0 END) AS null_keys FROM claims.analytics.dim_date
UNION ALL SELECT 'dim_member',   count(*), count(DISTINCT member_sk),   sum(CASE WHEN member_sk   IS NULL THEN 1 ELSE 0 END) FROM claims.analytics.dim_member
UNION ALL SELECT 'dim_hospital', count(*), count(DISTINCT hospital_sk), sum(CASE WHEN hospital_sk IS NULL THEN 1 ELSE 0 END) FROM claims.analytics.dim_hospital
UNION ALL SELECT 'dim_plan',     count(*), count(DISTINCT plan_sk),     sum(CASE WHEN plan_sk     IS NULL THEN 1 ELSE 0 END) FROM claims.analytics.dim_plan
UNION ALL SELECT 'fact_claims',  count(*), count(DISTINCT claim_id),    sum(CASE WHEN claim_id    IS NULL THEN 1 ELSE 0 END) FROM claims.analytics.fact_claims
ORDER BY tbl;

-- Orphan-join proof: a fact row pointing at a member / hospital / plan / date that does not exist must be zero
SELECT
  (SELECT count(*) FROM claims.analytics.fact_claims f LEFT JOIN claims.analytics.dim_member   m ON f.member_sk         = m.member_sk   WHERE m.member_sk   IS NULL) AS orphan_member_fk,
  (SELECT count(*) FROM claims.analytics.fact_claims f LEFT JOIN claims.analytics.dim_hospital h ON f.hospital_sk       = h.hospital_sk WHERE h.hospital_sk IS NULL) AS orphan_hospital_fk,
  (SELECT count(*) FROM claims.analytics.fact_claims f LEFT JOIN claims.analytics.dim_plan     p ON f.plan_sk             = p.plan_sk     WHERE p.plan_sk     IS NULL) AS orphan_plan_fk,
  (SELECT count(*) FROM claims.analytics.fact_claims f LEFT JOIN claims.analytics.dim_date     d ON f.admission_date_key = d.date_key    WHERE d.date_key    IS NULL) AS orphan_admission_fk,
  (SELECT count(*) FROM claims.analytics.fact_claims f LEFT JOIN claims.analytics.dim_date     d ON f.submitted_date_key = d.date_key    WHERE d.date_key    IS NULL) AS orphan_submitted_fk,
  (SELECT count(*) FROM claims.analytics.fact_claims f LEFT JOIN claims.analytics.dim_date     d ON f.settled_date_key   = d.date_key    WHERE d.date_key    IS NULL) AS orphan_settled_fk;